## Извлекаем датасет

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls -lh "/content/drive/MyDrive/CatBreedAI"

total 637M
-rw------- 1 root root  23K May 24 17:04 CatBreedAI.ipynb
-rw------- 1 root root 637M Apr 19 14:31 data.tar.gz


In [ ]:
!tar -xzf "/content/drive/MyDrive/CatBreedAI/data.tar.gz" -C /content/

In [ ]:
!ls /content

data  drive  sample_data


## Подготовка данных

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm
import os
from PIL import ImageFile, Image
from pathlib import Path
import shutil
import numpy as np

import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = '/content/data'
BATCH_SIZE = 64
IMG_SIZE = 224
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/cat_breed_models'

In [ ]:
ImageFile.LOAD_TRUNCATED_IMAGES = True
# warnings.filterwarnings('ignore', category=UserWarning, module='PIL')

class SafeImageFolder(datasets.ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        try:
            sample = self.loader(path)
            if self.transform is not None:
                sample = self.transform(sample)
            return sample, target
        except Exception as e:
            print(f"Error loading {path}: {e}, skipping...")
            # Возвращаем случайный другой элемент
            new_index = (index + 1) % len(self.samples)
            return self.__getitem__(new_index)

## Проверка файлов

In [ ]:
# import os
# from pathlib import Path
# from PIL import Image
# import torchvision.transforms as transforms

# # Трансформации
# transform = transforms.Compose([
#     transforms.RandomResizedCrop(384),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
# ])

# data_dir = '/content/data/train'

# for root, dirs, files in os.walk(data_dir):
#     for file in files:
#         if file.lower().endswith(('.jpg', '.jpeg', '.png')):
#             path = os.path.join(root, file)
#             try:
#                 with Image.open(path) as img:
#                     img = img.convert('RGB')
#                     # Применяем трансформации
#                     tensor = transform(img)
#             except Exception as e:
#                 print(f"❌ БИТЫЙ: {path}")
#                 print(f"   Ошибка: {e}")

In [ ]:
try:
  os.remove('/content/data/train/somali/33477513_41.jpg')
except:
  pass

In [ ]:
# Трансформации
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),  # Разнообразие масштаба
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),  # Иногда отражение по вертикали
    transforms.RandomRotation(degrees=20),  # Повороты ±20 градусов
    transforms.ColorJitter(
        brightness=0.3,   # Разная яркость
        contrast=0.3,     # Разный контраст
        saturation=0.3,   # Разная насыщенность
        hue=0.1           # Разный оттенок
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),  # Сдвиги
        scale=(0.9, 1.1),      # Масштаб
        shear=10                # Искажение
    ),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),  # Перспектива
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),  # Случайные прямоугольники
])

val_transform = transforms.Compose([
  transforms.Resize(int(IMG_SIZE * 1.14)),
  transforms.CenterCrop(IMG_SIZE),
  transforms.ToTensor(),
  transforms.Normalize(
      mean=[0.485, 0.456, 0.406],
      std=[0.229, 0.224, 0.225]
  )
])

# Загружаем список файлов
# train_dataset = datasets.ImageFolder(
#   root=os.path.join(DATA_DIR, 'train'),
#   transform=train_transform
# )

# val_dataset = datasets.ImageFolder(
#   root=os.path.join(DATA_DIR, 'val'),
#   transform=val_transform
# )
train_dataset = SafeImageFolder(
  root=os.path.join(DATA_DIR, 'train'),
  transform=train_transform
)

val_dataset = SafeImageFolder(
  root=os.path.join(DATA_DIR, 'val'),
  transform=val_transform
)

# DataLoader подгружает только текущий батч
train_loader = DataLoader(
  train_dataset, batch_size=BATCH_SIZE, shuffle=True,
  num_workers=0, pin_memory=True
)

val_loader = DataLoader(
  val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
  num_workers=0, pin_memory=True
)

In [ ]:
NUM_CLASSES = len(train_dataset.classes)
print(f"Классов: {NUM_CLASSES}")
print(f"Классы: {train_dataset.classes[:5]}...")

Классов: 66
Классы: ['abyssinian', 'american_bobtail', 'american_curl', 'american_shorthair', 'american_wirehair']...


## Модель
Используем предобученную `ConvNeXt-Tiny`

In [ ]:
DEVICE

device(type='cuda')

In [ ]:
model = timm.create_model(
    'convnext_tiny',  # Веса ImageNet-1K
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=0.0,
    drop_path_rate=0.1,
)

model = model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Разный lr для частей
optimizer = optim.AdamW(
  [
    {'params': model.stem.parameters(), 'lr': LEARNING_RATE * 0.01},
    {'params': model.stages.parameters(), 'lr': LEARNING_RATE * 0.05},
    {'params': model.head.parameters(), 'lr': LEARNING_RATE}  # Head учится быстрее
  ],
  weight_decay=0.1
)

# Динамический lr
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# Ускорение
if DEVICE == 'cuda':
  scaler = torch.amp.GradScaler('cuda')
else:
  scaler = None

## Подготовка к обучению

In [ ]:
def mixup_data(x, y, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)

    mixed_x = lam * x + (1-lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

In [ ]:
def train_one_epoch():
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # --- Debugging: Check label range ---
        if labels.min().item() < 0 or labels.max().item() >= NUM_CLASSES:
            print(f"\nERROR: Labels out of range! Min: {labels.min().item()}, Max: {labels.max().item()}, Expected: [0, {NUM_CLASSES-1}]")
            # You might want to break or raise an error here if this happens frequently
        # --- End Debugging ---

        images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.2)

        optimizer.zero_grad()

        if DEVICE == 'cuda':
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = lam * criterion(outputs, labels_a) + (1-lam) * criterion(outputs, labels_b)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        else:
            # Обычный проход для CPU
            outputs = model(images)
            loss = lam * criterion(outputs, labels_a) + (1-lam) * criterion(outputs, labels_b)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()

        # For accuracy calculation, we typically use the original labels or a combined approach for mixup.
        # Here, we'll simplify for now to get past the CUDA error, focusing on the loss issue.
        # The `predicted.eq(labels)` for mixup is not directly correct, as labels are mixed.
        # However, the primary issue is the CUDA error from loss, not accuracy calculation.
        # Let's keep `total` based on batch size and `correct` based on one of the labels for basic monitoring,
        # but acknowledge accuracy for mixup is more complex.
        _, predicted = outputs.max(1)
        total += labels.size(0)
        # For mixup, calculating 'correct' this way is an approximation and might not be precise,
        # but it serves to keep the progress bar updated. The real accuracy metric for mixup often
        # involves comparing with both labels_a and labels_b.
        correct += predicted.eq(labels_a).sum().item() # Using labels_a for a quick check

        pbar.set_postfix({
          'loss': f'{running_loss/(pbar.n+1):.3f}',
          'acc': f'{100.*correct/total:.1f}%'
        })

    return running_loss / len(train_loader), 100. * correct / total

In [ ]:
def validate():
  model.eval()
  running_loss = 0.0
  correct = 0
  total = 0

  with torch.no_grad():
    pbar = tqdm(val_loader, desc='Validation')
    for images, labels in pbar:
      images, labels = images.to(DEVICE), labels.to(DEVICE)
      images, labels_a, labels_b, lam = mixup_data(images, labels, alpha=0.2)

      with torch.cuda.amp.autocast():
        outputs = model(images)
        loss = lam * criterion(outputs, labels_a) + (1-lam) * criterion(outputs, labels_b)

      running_loss += loss.item()
      _, predicted = outputs.max(1)
      total += labels.size(0)
      correct += predicted.eq(labels).sum().item()

      pbar.set_postfix({
        'loss': f'{running_loss/(pbar.n+1):.3f}',
        'acc': f'{100.*correct/total:.1f}%'
      })

    return running_loss / len(val_loader), 100. * correct / total

## Обучение

In [ ]:
Path(DRIVE_BACKUP_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
best_acc = 0.0

for epoch in range(NUM_EPOCHS):
  print(f"\nEpoch: {epoch+1}/{NUM_EPOCHS}")

  train_loss, train_acc = train_one_epoch()
  val_loss, val_acc = validate()

  scheduler.step()

  print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
  print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

  # Сохраняем лучшую модель
  if val_acc > best_acc:
    best_acc = val_acc

    filename = 'best_cat_breed_model.pth'
    torch.save(
      {
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_acc':              val_acc,
        'classes':              train_dataset.classes
      },
      filename
    )
    shutil.copy2(
      os.path.join('/content', filename),
      os.path.join(DRIVE_BACKUP_DIR, filename)
    )


Epoch: 1/15


Validation: 100%|██████████| 75/75 [01:08<00:00,  1.09it/s, loss=2.371, acc=45.7%]


Train Loss: 2.8888 | Train Acc: 22.09%
Val Loss:   2.3710 | Val Acc:   45.66%

Epoch: 2/15


Validation: 100%|██████████| 75/75 [00:56<00:00,  1.33it/s, loss=2.228, acc=48.8%]


Train Loss: 2.5186 | Train Acc: 27.42%
Val Loss:   2.2276 | Val Acc:   48.77%

Epoch: 3/15


...

Validation: 100%|██████████| 75/75 [00:57<00:00,  1.31it/s, loss=2.026, acc=53.8%]


Train Loss: 2.1986 | Train Acc: 33.54%
Val Loss:   2.0259 | Val Acc:   53.80%

Epoch: 13/15


Validation: 100%|██████████| 75/75 [00:57<00:00,  1.30it/s, loss=2.021, acc=54.4%]


Train Loss: 2.1863 | Train Acc: 33.76%
Val Loss:   2.0209 | Val Acc:   54.36%

Epoch: 14/15


Training:  79%|███████▉  | 471/596 [15:34<04:12,  2.02s/it, loss=2.163, acc=31.9%]

In [ ]:
best_acc